In [3]:
!pip install snscrape
#This package will allow for the scraping of tweets in order to analyze the data

In [4]:
import snscrape.modules.twitter as sntwitter #scrape twitter tweets
import pandas as pd #create a Dataframe for the twitter data
import re #Tells python how to search for and manipulate text, in other words clean the tweets


In [5]:
import nltk #imports the NLTK library
nltk.download('vader_lexicon') #Downloads VADER sentiment dictionary
from nltk.sentiment.vader import SentimentIntensityAnalyzer #this analyzes the tone of a sentence used & whether it is positive, negative, or neutral


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


**DATA COLLECTION**

In [9]:
import pandas as pd

# Load uploaded CSV file (adjust the filename if needed)
df = pd.read_csv("/ai_sentiment_tweets_sample.csv")

# Preview
df.head()


#I originally tried to create a variable called ai_sentiment_terms to to filter tweets during scraping but received an error. I am now using a pre-selected dataset that has AI related tweets

,Date,Username,Text
0,2025-07-05,@AI_Guru,Artificial intelligence is reshaping every ind...
1,2025-06-17,@AI_Guru,I'm worried AI will replace too many jobs.
2,2025-07-03,@openai_fan,AI-generated art is so impressive.
3,2025-06-22,@elonfan,AI is going to take over the world!
4,2025-06-28,@AI_Guru,I love how AI makes my life easier.


**CLEANING TWEET TEXT FOR SENTIMENT ANALYSIS**

In [10]:
# Define a function to clean tweet text
def clean_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # Remove URLs
    text = re.sub(r'@\w+|\#', '', text)                 # Remove mentions and hashtags
    text = re.sub(r'[^\w\s]', '', text)                 # Remove punctuation
    return text.lower()                                 # Convert to lowercase

# Apply the function to create a cleaned text column
df['Cleaned_Text'] = df['Text'].apply(clean_text)

# Preview original vs cleaned text
df[['Text', 'Cleaned_Text']].head()


,Text,Cleaned_Text
0,Artificial intelligence is reshaping every ind...,artificial intelligence is reshaping every ind...
1,I'm worried AI will replace too many jobs.,im worried ai will replace too many jobs
2,AI-generated art is so impressive.,aigenerated art is so impressive
3,AI is going to take over the world!,ai is going to take over the world
4,I love how AI makes my life easier.,i love how ai makes my life easier


**APPLY VADER SENTIMENT ANALYSIS USING NLTK**

In [11]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

#Remember that VADER is built into NLTK to generate a sentiment score

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [14]:
# Initialize the VADER sentiment analyzer
sid = SentimentIntensityAnalyzer()

# Apply VADER to the cleaned tweet text
df['Sentiment_Score'] = df['Cleaned_Text'].apply(lambda x: sid.polarity_scores(x)['compound'])

df[['Cleaned_Text', 'Sentiment_Score']].head() #this will show the first 5 rows of the Cleaned_Text and Sentiment_Score columns


,Cleaned_Text,Sentiment_Score
0,artificial intelligence is reshaping every ind...,0.4767
1,im worried ai will replace too many jobs,-0.2960
2,aigenerated art is so impressive,0.6418
3,ai is going to take over the world,0.0000
4,i love how ai makes my life easier,0.7906


It's important to note that the compound score is a sentiment score between -1 (very negative) and 1 (very positive)

**Classify sentiment scores**

In [15]:
# Define a function to label the sentiment
def get_sentiment_label(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

# Apply the labeling function
df['VADER_Label'] = df['Sentiment_Score'].apply(get_sentiment_label)

# Preview results
df[['Cleaned_Text', 'Sentiment_Score', 'VADER_Label']].head()


,Cleaned_Text,Sentiment_Score,VADER_Label
0,artificial intelligence is reshaping every ind...,0.4767,Positive
1,im worried ai will replace too many jobs,-0.2960,Negative
2,aigenerated art is so impressive,0.6418,Positive
3,ai is going to take over the world,0.0000,Neutral
4,i love how ai makes my life easier,0.7906,Positive


**My own human judgement and evaluation**

In [17]:
# Select a random sample of 30 tweets to evaluate manually
sample_df = df[['Text', 'Cleaned_Text', 'Sentiment_Score', 'VADER_Label']].sample(30, random_state=1)

# Save this sample so you can open it and label manually
sample_df.to_csv("vader_sample_for_manual_labeling.csv", index=False)

sample_df #This will show all 30 random tweets from the entire DataFrame



,Text,Cleaned_Text,Sentiment_Score,VADER_Label
304,AI is going to take over the world!,ai is going to take over the world,0.0000,Neutral
340,AI is going to take over the world!,ai is going to take over the world,0.0000,Neutral
47,AI is going to take over the world!,ai is going to take over the world,0.0000,Neutral
67,AI-generated art is so impressive.,aigenerated art is so impressive,0.6418,Positive
479,AI-generated art is so impressive.,aigenerated art is so impressive,0.6418,Positive
485,AI-generated art is so impressive.,aigenerated art is so impressive,0.6418,Positive
310,I love how AI makes my life easier.,i love how ai makes my life easier,0.7906,Positive
31,I love how AI makes my life easier.,i love how ai makes my life easier,0.7906,Positive
249,Artificial intelligence is reshaping every ind...,artificial intelligence is reshaping every ind...,0.4767,Positive
90,The future of AI is both exciting and scary.,the future of ai is both exciting and scary,0.0000,Neutral


In [20]:
sample_df.to_csv("vader_sample_for_manual_labeling.csv", index=False)
#This will export the csv for labeling

Show the full table of tweets with both the VADER and my
 human sentiment labels side by side.



In [27]:
import pandas as pd

# Load your manually labeled CSV
labeled_df = pd.read_csv("/vader_sample_for_manual_labeling.csv")

# Preview the comparison
labeled_df[['Text', 'VADER_Label', 'Human_Label']]


,Text,VADER_Label,Human_Label
0,AI is going to take over the world!,Neutral,Negative
1,AI is going to take over the world!,Neutral,Negative
2,AI is going to take over the world!,Neutral,Negative
3,AI-generated art is so impressive.,Positive,Positive
4,AI-generated art is so impressive.,Positive,Positive
5,AI-generated art is so impressive.,Positive,Positive
6,I love how AI makes my life easier.,Positive,Positive
7,I love how AI makes my life easier.,Positive,Positive
8,Artificial intelligence is reshaping every ind...,Positive,Neutral
9,The future of AI is both exciting and scary.,Neutral,Neutral


Calculate VADER accuracy

In [28]:
# Calculate the percentage of matching labels
accuracy = (labeled_df['VADER_Label'] == labeled_df['Human_Label']).mean()
print(f"VADER Accuracy: {accuracy * 100:.2f}%")


VADER Accuracy: 53.33%


Show disagreements

In [29]:
# Show only the rows where VADER and Human labels don't match
disagreements = labeled_df[labeled_df['VADER_Label'] != labeled_df['Human_Label']]
disagreements[['Text', 'VADER_Label', 'Human_Label']]


,Text,VADER_Label,Human_Label
0,AI is going to take over the world!,Neutral,Negative
1,AI is going to take over the world!,Neutral,Negative
2,AI is going to take over the world!,Neutral,Negative
8,Artificial intelligence is reshaping every ind...,Positive,Neutral
11,AI is going to take over the world!,Neutral,Negative
14,Artificial intelligence is reshaping every ind...,Positive,Neutral
16,ChatGPT helped me write my essay!,Neutral,Positive
17,ChatGPT helped me write my essay!,Neutral,Positive
18,AI is going to take over the world!,Neutral,Negative
19,AI is overrated. Still needs human input.,Neutral,Negative
